# HC-MARL: Real Safety-Gymnasium Benchmark

Trains and evaluates on **real** Safety-Gymnasium environments (not simulated mocks).
Compares ECBF-PPO vs CPO vs PPO-Lagrangian vs FOCOPS on standard benchmarks.

**Environments:** SafetyPointGoal1-v0, SafetyAntVelocity-v1

**References:**
- Ji et al. "Safety-Gymnasium" (NeurIPS 2023 Datasets Track)
- Ji et al. "OmniSafe" (arXiv 2305.09304)

**Requirements:** Colab Pro with T4 GPU

## Step 1: Install

In [ ]:
# Install dependencies
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install gymnasium scipy cvxpy numpy pandas matplotlib pyyaml wandb
!pip install safety-gymnasium omnisafe
print('All packages installed.')

## Step 2: Clone

In [ ]:
import os

REPO_URL = "https://github.com/ADITYA-WORK-MAITI/hcmarl-project.git"
PROJECT_DIR = "/content/hcmarl_project"

if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    !cd {PROJECT_DIR} && git pull

os.chdir(PROJECT_DIR)
!pip install -e . 2>/dev/null || echo 'Package install skipped'

# Verify
!python -c "import hcmarl; print('HC-MARL package OK')"
!python -c "import safety_gymnasium; print('Safety-Gymnasium OK')"
!python -c "import omnisafe; print('OmniSafe OK')"
!python -c "import torch; print(f'PyTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}')"

## Step 3: Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = "/content/drive/MyDrive/hcmarl_safety_gym_results"
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Results will be saved to: {DRIVE_DIR}')

## Step 4: W&B

In [ ]:
import wandb
try:
    wandb.login()
    print('W&B logged in successfully')
except Exception as e:
    print(f'W&B login skipped: {e}')

## GPU Check

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU! Enable in Runtime > Change runtime type'
gpu = torch.cuda.get_device_name(0)
mem = torch.cuda.get_device_properties(0).total_mem / 1e9
print(f'GPU: {gpu} ({mem:.1f} GB)')

## Training: OmniSafe Benchmarks

Run PPOLag, CPO, FOCOPS on SafetyPointGoal1-v0 and SafetyAntVelocity-v1.
5 seeds each, 1M steps per seed.

In [ ]:
from hcmarl.baselines.omnisafe_wrapper import run_omnisafe_benchmark

ENVS = ['SafetyPointGoal1-v0', 'SafetyAntVelocity-v1']
ALGOS = ['PPOLag', 'CPO', 'FOCOPS']
SEEDS = [0, 1, 2, 3, 4]
TOTAL_STEPS = 1_000_000

all_results = {}
for env_id in ENVS:
    for algo in ALGOS:
        key = f'{algo}_{env_id}'
        print(f'\n{"="*70}')
        print(f'Running: {key}')
        print(f'{"="*70}')
        results = run_omnisafe_benchmark(
            algo=algo, env_id=env_id, seeds=SEEDS,
            total_steps=TOTAL_STEPS, device='cuda',
            log_dir=os.path.join(DRIVE_DIR, 'omnisafe_logs'),
        )
        all_results[key] = results
        print(f'Done: {key} | mean_reward={results["aggregate"]["mean_reward"]:.2f}')

## Training: HC-MARL ECBF Filter on Safety-Gymnasium

In [ ]:
from hcmarl.envs.safety_gym_real import run_safety_gym_benchmark

for env_id in ENVS:
    print(f'\nECBF benchmark on {env_id}:')
    ecbf_results = run_safety_gym_benchmark(
        env_id=env_id, n_episodes=100, max_steps=1000, verbose=True
    )
    all_results[f'ECBF_{env_id}'] = ecbf_results

## Results Summary

In [ ]:
import pandas as pd
import json

# Build comparison table
rows = []
for key, res in all_results.items():
    if 'aggregate' in res:
        rows.append({
            'Method': key,
            'Mean Reward': res['aggregate']['mean_reward'],
            'Std Reward': res['aggregate']['std_reward'],
            'Mean Cost': res['aggregate']['mean_cost'],
            'Std Cost': res['aggregate']['std_cost'],
        })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

# Save
results_path = os.path.join(DRIVE_DIR, 'safety_gym_comparison.json')
with open(results_path, 'w') as f:
    json.dump(all_results, f, indent=2, default=str)
print(f'\nResults saved to: {results_path}')

## Save All to Drive

In [ ]:
import shutil
for folder in ['omnisafe_results', 'logs', 'results']:
    src = os.path.join(PROJECT_DIR, folder)
    dst = os.path.join(DRIVE_DIR, folder)
    if os.path.exists(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print(f'Copied {folder}/ to Drive')
print('\nAll results saved to Google Drive!')